# Projekt: Klasyfikacja i analiza obrazow kwiatow z uzyciem transfer learningu

**Autor:** Daniel Stefanski  
**Przedmiot:** Sieci neuronowe  
**Kod projektu:** **Projekt_Daniel_Stefanski.py**

Celem projektu jest zaprojektowanie, zaimplementowanie i przeanalizowanie modelu glebokiego uczenia do praktycznego zadania klasyfikacji obrazow. W projekcie wykorzystano konwolucyjna siec neuronowa w podejsciu transfer learningu, gdzie baza modelu **MobileNetV2** zostala uzyta jako ekstraktor cech, a na jej wyjsciu dodano wlasna glowice klasyfikacyjna dla pieciu klas kwiatow.

Finalne wyniki opisane w dokumentacji pochodza z pelnego uruchomienia programu wykonanego 01.06.2026. Program zapisal wyniki eksperymentow, wykresy, macierze pomylek, przykladowe bledne predykcje oraz informacje o srodowisku uruchomieniowym w folderze **wyniki**.

## 1. Uzasadnienie wyboru tematu

Wybrano klasyfikacje i analize obrazow, poniewaz jest to jeden z glownych obszarow zastosowan glebokiego uczenia. Problem dobrze nawiazuje do tematow realizowanych podczas laboratoriow, szczegolnie do klasyfikacji obrazow, sieci CNN, augmentacji danych oraz transfer learningu.

Zamiast trenowac siec od zera, zastosowano model **MobileNetV2** wytrenowany wczesniej na duzym zbiorze ImageNet. Takie podejscie jest praktyczne, poniewaz pozwala wykorzystac gotowe cechy wizualne, np. krawedzie, tekstury i ksztalty, a nastepnie dostosowac model do konkretnego problemu klasyfikacji kwiatow.

## 2. Dane

W projekcie wykorzystano zbior **flower_photos**, zawierajacy obrazy pieciu klas kwiatow:

- **daisy**,
- **dandelion**,
- **roses**,
- **sunflowers**,
- **tulips**.

Zbior zostal skopiowany do folderu projektu jako **flower_photos**, dzieki czemu eksperymenty nie wymagaja ponownego pobierania danych. Obrazy zostaly podzielone stratifikacyjnie na zbiory treningowy, walidacyjny i testowy.

In [ ]:
from pathlib import Path
import json
import pandas as pd
from IPython.display import Image, display

BASE_DIR = Path.cwd()
RESULTS_DIR = BASE_DIR / "wyniki"

with open(RESULTS_DIR / "dataset_summary.json", encoding="utf-8") as f:
    dataset_summary = json.load(f)

rows = []
for split_name, split_data in dataset_summary["splits"].items():
    row = {"split": split_name, "total": split_data["total"]}
    row.update(split_data["class_counts"])
    rows.append(row)

pd.DataFrame(rows)

## 2.1. Srodowisko uruchomieniowe

Skrypt zapisuje informacje o uruchomieniu do pliku **wyniki/run_info.json**. Dzieki temu mozna sprawdzic, kiedy wykonano trening, jaka wersja TensorFlow zostala uzyta oraz czy program korzystal z GPU. W finalnym uruchomieniu TensorFlow nie wykryl GPU, dlatego trening zostal wykonany na CPU.

In [ ]:
with open(RESULTS_DIR / "run_info.json", encoding="utf-8") as f:
    run_info = json.load(f)

pd.DataFrame([
    {
        "data uruchomienia": run_info["run_datetime"],
        "tryb szybki": run_info["quick_mode"],
        "Python": run_info["python_version"],
        "TensorFlow": run_info["tensorflow_version"],
        "GPU dostepne": run_info["gpu_available"],
        "urzadzenia GPU": ", ".join(run_info["gpu_devices"]) if run_info["gpu_devices"] else "brak",
        "image_size": run_info["image_size"],
        "batch_size": run_info["batch_size"],
    }
])

## 3. Przygotowanie danych

Wszystkie obrazy zostaly przeskalowane do rozmiaru **160 x 160** pikseli. Dane treningowe byly dodatkowo mieszane oraz przetwarzane wsadowo. W czesci eksperymentow zastosowano augmentacje danych:

- losowe odbicie poziome,
- losowa rotacja,
- losowe przyblizenie,
- losowa zmiana kontrastu.

Augmentacja ma ograniczac przeuczenie modelu i zwiekszac odpornosc klasyfikatora na naturalne roznice miedzy zdjeciami.

## 4. Architektura modelu

Zastosowany model sklada sie z dwoch glownych czesci:

1. Bazy **MobileNetV2** bez oryginalnej warstwy klasyfikacyjnej (**include_top=False**).
2. Wlasnej glowicy klasyfikacyjnej:
   - **GlobalAveragePooling2D**,
   - **Dropout**,
   - warstwa **Dense** z aktywacja **ReLU**,
   - kolejny **Dropout**,
   - warstwa wyjsciowa **Dense** z aktywacja **softmax**.

Model uczony byl z funkcja straty **sparse_categorical_crossentropy**. Do optymalizacji wykorzystano algorytm **Adam**, ktory aktualizuje wagi sieci na podstawie gradientow obliczanych metoda propagacji wstecznej.

## 5. Eksperymenty

Przeprowadzono cztery uruchomienia, ktore pozwalaja porownac trzy glowne aspekty uczenia:

- wplyw learning rate,
- wplyw augmentacji danych,
- wplyw fine-tuningu ostatnich warstw modelu bazowego.

Eksperymenty:

- **exp1_lr_1e-4**: model z augmentacja, learning rate **1e-4**, bez fine-tuningu,
- **exp1_lr_1e-3**: model z augmentacja, learning rate **1e-3**, bez fine-tuningu,
- **exp2_without_augmentation**: model bez augmentacji, learning rate **1e-4**,
- **exp3_fine_tuning**: model z augmentacja i dodatkowym fine-tuningiem ostatnich warstw **MobileNetV2**.

In [ ]:
with open(RESULTS_DIR / "comparison.json", encoding="utf-8") as f:
    comparison = json.load(f)

comparison_df = pd.DataFrame(comparison)
comparison_df[[
    "name",
    "learning_rate",
    "augmentation",
    "fine_tune",
    "best_val_accuracy",
    "test_accuracy",
    "test_loss",
    "wrong_predictions_count",
    "mean_confidence",
]]

In [ ]:
display(Image(filename=str(RESULTS_DIR / "comparison_accuracy.png")))

## 6. Interpretacja wynikow eksperymentow

Najlepszy wynik na zbiorze testowym uzyskal eksperyment **exp1_lr_1e-3**, dla ktorego **test_accuracy** wynioslo okolo **0.8984**. Ten wariant mial rowniez najwyzsza srednia pewnosc predykcji oraz najmniejsza liczbe blednych klasyfikacji sposrod porownywanych modeli: **56** bledow na **551** obrazow testowych.

Wariant z learning rate **1e-4** uczyl sie ostrozniej i osiagnal nizsza dokladnosc testowa, okolo **0.8548**. Model bez augmentacji uzyskal wynik zblizony do wariantu **1e-4**, czyli okolo **0.8584**, ale jego lepszy wynik od wariantu z augmentacja przy tym samym learning rate moze wynikac z ograniczonej liczby epok lub zbyt silnej augmentacji.

Fine-tuning nie poprawil wyniku testowego w tym uruchomieniu. Model **exp3_fine_tuning** osiagnal accuracy okolo **0.8548**. Oznacza to, ze przy dostepnym czasie treningu i dobranych parametrach odmrozenie ostatnich warstw nie dalo przewagi nad zamrozona baza **MobileNetV2**. Najbardziej widoczny pozytywny wplyw mial natomiast learning rate **1e-3**, ktory pozwolil modelowi uczyc sie szybciej i skuteczniej niz wariant **1e-4**.

## 7. Szczegolowa analiza najlepszego modelu

Za najlepszy model przyjeto **exp1_lr_1e-3**, poniewaz uzyskal najwyzsza dokladnosc testowa i najmniej blednych predykcji. Ponizej przedstawiono krzywe uczenia, macierz pomylek oraz przykladowe bledne klasyfikacje.

In [ ]:
BEST_EXP = "exp1_lr_1e-3"
best_dir = RESULTS_DIR / BEST_EXP

display(Image(filename=str(best_dir / "history.png")))
display(Image(filename=str(best_dir / "confusion_matrix.png")))
display(Image(filename=str(best_dir / "wrong_predictions.png")))

In [ ]:
with open(best_dir / "results.json", encoding="utf-8") as f:
    best_results = json.load(f)

report = pd.DataFrame(best_results["classification_report"]).T
report

Macierz pomylek pozwala ocenic, ktore klasy byly dla modelu najtrudniejsze. W klasyfikacji kwiatow pomylki sa naturalne, poniewaz niektore klasy moga miec podobne kolory, ksztalty platkow albo tlo. Przykladowe bledne predykcje pokazuja, ze model czesto myli sie na zdjeciach mniej typowych, gorzej skadrowanych lub zawierajacych elementy utrudniajace rozpoznanie dominujacego kwiatu.

## 8. Odniesienie do metod z sylabusa

Projekt wykorzystuje kilka kluczowych elementow omawianych w ramach przedmiotu:

- konwolucyjne sieci neuronowe do analizy obrazow,
- transfer learning z modelem **MobileNetV2**,
- propagacje wsteczna do aktualizacji wag,
- optymalizator **Adam**,
- augmentacje danych jako metode ograniczania przeuczenia,
- ewaluacje modelu na osobnym zbiorze testowym,
- analize wynikow przez metryki, wykresy, macierz pomylek i bledne predykcje.

## 9. Ograniczenia projektu

Projekt ma kilka ograniczen:

- wykorzystany zbior danych jest stosunkowo niewielki,
- liczba klas jest ograniczona do pieciu gatunkow kwiatow,
- fine-tuning byl wykonany tylko dla ostatnich warstw modelu bazowego,
- nie przeprowadzono bardzo szerokiego strojenia hiperparametrow,
- finalny trening zostal wykonany na CPU, poniewaz TensorFlow w tym srodowisku nie wykryl GPU,
- wyniki moga zalezec od losowego podzialu danych i liczby epok.

Potencjalne ulepszenia obejmuja zastosowanie wiekszego zbioru danych, dluzszy trening, test innych architektur, np. **EfficientNetB0**, oraz dokladniejsza analize interpretowalnosci modelu, np. przez **Grad-CAM**.

## 10. Podsumowanie

W projekcie zaimplementowano system klasyfikacji obrazow kwiatow z wykorzystaniem transfer learningu. Model oparty na **MobileNetV2** zostal przetestowany w kilku wariantach eksperymentalnych. Najlepszy wynik uzyskano dla konfiguracji z augmentacja i learning rate **1e-3**, gdzie dokladnosc na zbiorze testowym wyniosla okolo **89.84%**.

Wyniki pokazuja, ze transfer learning pozwala uzyskac dobra jakosc klasyfikacji nawet przy umiarkowanie duzym zbiorze danych. Jednoczesnie eksperymenty potwierdzaja, ze hiperparametry, takie jak learning rate, moga miec duzy wplyw na koncowa skutecznosc modelu.

**Link do pliku ze zdjęciami:** https://drive.google.com/file/d/1CTnBvCDMevzTvchc28JLl0bNMdOnxNnU/view?usp=sharing

**Link do repozytorium GitHub:** https://github.com/Daniel-Stefanski/Sieci_Neuronowe/tree/main/Projekt

## 11. Zrzuty z terminala

Ponizej dolaczono log tekstowy oraz zrzuty terminala dokumentujace wykonanie pelnego treningu i zapis wynikow. Skrypt automatycznie zapisuje aktualny przebieg do pliku **terminal/latest_run_log.txt**. Zrzuty ekranu pozostaja dodatkiem wizualnym.

In [ ]:
log_path = BASE_DIR / "terminal" / "latest_run_log.txt"
fallback_log_path = RESULTS_DIR / "logi" / "latest_run_log.txt"

if not log_path.exists() and fallback_log_path.exists():
    log_path = fallback_log_path

if log_path.exists():
    log_text = log_path.read_text(encoding="utf-8", errors="replace")
    print(log_text[-6000:])
else:
    print("Brak automatycznego logu tekstowego. Uruchom skrypt ponownie, aby wygenerowac terminal/latest_run_log.txt.")

In [ ]:
terminal_dir = BASE_DIR / "terminal"
for image_path in sorted(terminal_dir.glob("*.png")):
    display(Image(filename=str(image_path)))